In [1]:
# @title Cell 1: Mount Drive & Imports
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import drive

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

drive.mount('/content/drive')

Using device: cuda
Mounted at /content/drive


In [2]:
# @title Cell 2: Load Data & Create Sequences
DATA_PATH = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 1/Data Pipeline/data/processed/crude_weekly_complete.parquet'

df = pd.read_parquet(DATA_PATH)

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

TARGET_COL = 'target_next_week_change'
IGNORE_COLS = ['date', TARGET_COL, 'event_name', 'stocks_change_outlier']
FEATURE_COLS = [col for col in df.columns if col not in IGNORE_COLS]

df_clean = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).reset_index(drop=True)

# Define test holdout size & lookback window (e.g., 4 weeks of history to predict next week)
TEST_SIZE = 104
LOOKBACK = 4

# Scale features (Fit ONLY on train portion to avoid data leakage)
train_raw = df_clean.iloc[:-TEST_SIZE].copy()
test_raw = df_clean.iloc[-TEST_SIZE - LOOKBACK:].copy()  # Include lookback buffer for test start

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

train_features_scaled = feature_scaler.fit_transform(train_raw[FEATURE_COLS])
train_target_scaled = target_scaler.fit_transform(train_raw[[TARGET_COL]])

test_features_scaled = feature_scaler.transform(test_raw[FEATURE_COLS])
test_target_scaled = target_scaler.transform(test_raw[[TARGET_COL]])

# Helper function to create sliding sequences
def create_sequences(features, targets, lookback):
    X, y = [], []
    for i in range(len(features) - lookback):
        X.append(features[i : i + lookback])
        y.append(targets[i + lookback])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_features_scaled, train_target_scaled, LOOKBACK)
X_test, y_test_scaled = create_sequences(test_features_scaled, test_target_scaled, LOOKBACK)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

print(f"Sequences Created -> Train X shape: {X_train_t.shape}, Test X shape: {X_test_t.shape}")

Sequences Created -> Train X shape: torch.Size([1063, 4, 19]), Test X shape: torch.Size([104, 4, 19])


In [3]:
# @title Cell 3: Define Regularized LSTM Model
class CrudeLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, dropout=0.3):
        super(CrudeLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Apply dropout to the last sequence step output before final prediction
        out = self.dropout(lstm_out[:, -1, :])
        out = self.fc(out)
        return out

# Small hidden dim (16) prevents capacity for noise memorization
model = CrudeLSTM(input_dim=len(FEATURE_COLS), hidden_dim=16, dropout=0.3).to(device)
criterion = nn.MSELoss()

# Higher weight decay (1e-3) forces weights towards zero
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

print(model)

CrudeLSTM(
  (lstm): LSTM(19, 16, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)


In [4]:
# @title Cell 4: Train Regularized LSTM & Evaluate
# Reduced epochs from 80 -> 25 to stop early before memorizing training set
EPOCHS = 25

model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_X.size(0)

    avg_loss = epoch_loss / len(train_loader.dataset)
    if epoch % 5 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    preds_scaled = model(X_test_t).cpu().numpy()

# Inverse scale predictions back to original crude oil barrel scale (kb)
preds = target_scaler.inverse_transform(preds_scaled).flatten()
y_test_actual = df_clean.iloc[-TEST_SIZE:][TARGET_COL].values

mae = mean_absolute_error(y_test_actual, preds)
rmse = np.sqrt(mean_squared_error(y_test_actual, preds))
r2 = r2_score(y_test_actual, preds)
dir_acc = np.mean(np.sign(preds) == np.sign(y_test_actual)) * 100

print("\n=== Regularized LSTM Out-of-Sample Metrics (104-Week Test) ===")
print(f"MAE : {mae:.2f} kb")
print(f"RMSE: {rmse:.2f} kb")
print(f"R²  : {r2:.4f}")
print(f"Directional Accuracy: {dir_acc:.1f}%")

Epoch 5/25 - Loss: 0.9159
Epoch 10/25 - Loss: 0.8478
Epoch 15/25 - Loss: 0.8183
Epoch 20/25 - Loss: 0.7779
Epoch 25/25 - Loss: 0.7517

=== Regularized LSTM Out-of-Sample Metrics (104-Week Test) ===
MAE : 3941.76 kb
RMSE: 4910.40 kb
R²  : -0.0055
Directional Accuracy: 51.0%


In [5]:
# @title Cell 5: Save Outputs directly to Google Drive
import os

OUTPUT_DIR = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.4 LSTM model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save Metrics
metrics_df = pd.DataFrame([{
    'Model': 'LSTM (PyTorch Regularized)',
    'MAE (kb)': round(mae, 2),
    'RMSE (kb)': round(rmse, 2),
    'R² Score': round(r2, 4),
    'Directional Accuracy (%)': round(dir_acc, 2)
}])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'lstm_metrics.csv'), index=False)

# 2. Save Predictions
results_df = df_clean.iloc[-TEST_SIZE:][['date', TARGET_COL]].copy() if 'date' in df_clean.columns else df_clean.iloc[-TEST_SIZE:][[TARGET_COL]].copy()
results_df['LSTM_Prediction'] = preds
results_df.to_csv(os.path.join(OUTPUT_DIR, 'lstm_predictions.csv'), index=False)

print(f"All files successfully saved to Drive at:\n{OUTPUT_DIR}")

All files successfully saved to Drive at:
/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.4 LSTM model


In [7]:
import torch
import joblib
import json
import os

# 1. Define Drive Output Directory for LSTM
LSTM_DIR = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.4 LSTM model'
os.makedirs(LSTM_DIR, exist_ok=True)

# 2. Save Trained PyTorch Model Weights & Metadata
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': len(FEATURE_COLS),
    'hidden_dim': 16,
    'dropout': 0.3,
    'num_layers': 1
}, os.path.join(LSTM_DIR, 'lstm_model.pt'))

# 3. Save Input Feature Scaler (scaler / feature_scaler)
scaler_to_save = scaler if 'scaler' in globals() else feature_scaler
joblib.dump(scaler_to_save, os.path.join(LSTM_DIR, 'lstm_feature_scaler.pkl'))

# 4. Target Scaling Parameters Fallback Logic
if 'y_train_raw' in globals():
    target_series = y_train_raw
elif 'y_train' in globals():
    target_series = y_train
else:
    target_series = y

scaling_stats = {
    'y_mean': float(target_series.mean()),
    'y_std': float(target_series.std())
}

with open(os.path.join(LSTM_DIR, 'scaling_stats.json'), 'w') as f:
    json.dump(scaling_stats, f)

print(f"Successfully saved lstm_model.pt, lstm_feature_scaler.pkl, and scaling_stats.json to:\n{LSTM_DIR}")

Successfully saved lstm_model.pt, lstm_feature_scaler.pkl, and scaling_stats.json to:
/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.4 LSTM model
